# 07 — End-to-End Pipeline

Runs the full production path — `InventoryPipeline.run_with_trace()` — on real images, visualizing every intermediate stage (Detection -> Overlap -> Refinement -> Crop -> per-crop Retrieval/Decision/Plugins/Rerank), then exports results through the real `StorageManager` (never a notebook-local reimplementation of JSON/CSV/image export).

Covered in this notebook:

1. Setup and helpers
2. Load `InventoryPipeline` and `StorageManager`
3. Run `run_with_trace()` on one image
4. Detection stage — all raw boxes
5. Overlap stage — suspicious pairs and connected groups
6. Refinement stage — original vs. refined boxes, fallback flags
7. Per-crop trace table — Retrieval -> Decision -> Plugins -> Final
8. Final annotated result, JSON, and CSV via the real `StorageManager`
9. Batch run + per-stage latency breakdown
10. Summary

**Color convention** (consistent with earlier notebooks): green = accepted prediction, orange = uncertain, gray = rejected/fallback, red dashed = ground truth (when shown).

## 1. Environment setup

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
assert (PROJECT_ROOT / "configs" / "config.yaml").is_file()

In [ ]:
import json
import random

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

from src.core.config import load_config
from src.core.utils import generate_id, list_image_files, load_image_bgr
from src.models.models import ImageData
from src.pipeline.pipeline import InventoryPipeline
from src.storage.results import StorageManager

pd.set_option("display.max_colwidth", 80)
%matplotlib inline

STATUS_COLOR = {"accepted": "#55A868", "uncertain": "#DD8452", "rejected": "#888888"}


def show_image_bgr(ax, image_array_bgr, title=""):
    ax.imshow(cv2.cvtColor(image_array_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis("off")


IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

## 2. Config overview

In [ ]:
config = load_config()

print(f"Detection backend:    {config.detection.backend}")
print(f"Refinement backend:   {config.refinement.backend}  (enabled={config.refinement.enabled})")
print(f"Retrieval backend:    {config.retrieval.backend}")
print(f"Output dir:           {config.resolve_path(config.paths.output_dir)}")
print(f"  json_filename:      {config.storage.json_filename}")
print(f"  csv_filename:       {config.storage.csv_filename}")
print(f"  annotated_filename: {config.storage.annotated_image_filename}")
print(f"\nOverlap trigger: iou_threshold={config.refinement.trigger.iou_threshold} "
      f"overlap_ratio_threshold={config.refinement.trigger.overlap_ratio_threshold}")

## 3. Load `InventoryPipeline` and `StorageManager`

Mirrors `src/inference/infer.py`'s `InferenceRunner.__init__` exactly — both are constructed once and reused. `run_with_trace()` is only exposed on `InventoryPipeline` directly (not through `InferenceRunner`, which only calls `run()`), so this notebook holds the pipeline instance itself rather than wrapping it.

In [ ]:
try:
    pipeline = InventoryPipeline(config)
    print("InventoryPipeline loaded (Detector, OverlapResolver, Refiner, Cropper, "
          "Retriever, DecisionEngine, PluginManager, Reranker all initialized once).")
except FileNotFoundError as exc:
    print(f"InventoryPipeline could not be loaded: {exc}")
    print("The gallery index/metadata is likely missing — build it first, "
          "e.g. via 04_retrieval_analysis.ipynb §2, then re-run this cell.")
    pipeline = None

storage = StorageManager(config) if pipeline is not None else None

## 4. Run `run_with_trace()` on one image

Uses `load_image_bgr` from `src.core.utils` — the exact loader `InferenceRunner.run_single` uses — rather than a notebook-local image reader.

In [ ]:
benchmark_images_dir = config.resolve_path(config.paths.benchmark_images_dir)
query_dir = config.resolve_path(config.paths.query_dir)
candidate_dir = benchmark_images_dir if benchmark_images_dir.exists() and any(
    benchmark_images_dir.iterdir()
) else query_dir
available_images = sorted([f for f in candidate_dir.iterdir() if f.suffix.lower() in IMAGE_EXTS])
assert available_images, f"No images found in {candidate_dir}"

sample_path = random.choice(available_images)
image_array = load_image_bgr(sample_path)
height, width = image_array.shape[:2]
sample_image = ImageData(
    image_id=generate_id(prefix="img_"),
    source_path=str(sample_path),
    image_array=image_array,
    width=width,
    height=height,
)

print(f"Sample image: {sample_path.name}  ({width}x{height})")

if pipeline is not None:
    result, trace = pipeline.run_with_trace(sample_image)
    print(f"\nInventoryResult: {result.total_items} item(s) accepted, "
          f"{result.processing_time_ms:.2f} ms total")
else:
    result, trace = None, None
    print("Pipeline not available — skip.")

## 5. Detection stage — all raw boxes

Every box `Detector` produced, before overlap/refinement/decision. Box color reflects detector confidence relative to `detection.confidence_threshold` — this stage has no accept/uncertain/reject concept yet, only the raw detector output.

In [ ]:
if trace is not None:
    fig, ax = plt.subplots(figsize=(9, 9 * height / width))
    show_image_bgr(ax, sample_image.image_array,
                    title=f"Detection stage: {len(trace.detection_result.detections)} box(es) "
                          f"({trace.detection_result.processing_time_ms:.1f} ms)")

    for i, det in enumerate(trace.detection_result.detections):
        b = det.bbox
        rect = mpatches.Rectangle((b.x1, b.y1), b.width, b.height, fill=False,
                                    edgecolor="#4C72B0", linewidth=1.5)
        ax.add_patch(rect)
        ax.text(b.x1, b.y1 - 3, f"#{i} {det.confidence:.2f}", color="#4C72B0", fontsize=7)

    plt.tight_layout()
    plt.show()
else:
    print("No trace available — skip.")

## 6. Overlap stage — suspicious pairs and connected groups

Each `OverlapGroup` gets its own color; detections not in any group are drawn in gray. Uses `OverlapResolver`'s own DTOs (`OverlapResult.groups`) directly — never a notebook-local re-derivation of which boxes overlap (`find_suspicious_pairs` is the single source of truth per `src/pipeline/overlap.py`).

In [ ]:
if trace is not None:
    overlap_result = trace.overlap_result
    print(f"needs_refinement={overlap_result.needs_refinement}  "
          f"suspicious_pairs={len(overlap_result.pairs)}  groups={overlap_result.group_count}  "
          f"({overlap_result.processing_time_ms:.1f} ms)")

    group_colors = plt.cm.tab10(np.linspace(0, 1, max(len(overlap_result.groups), 1)))
    detection_to_group = {}
    for g_idx, group in enumerate(overlap_result.groups):
        for det_idx in group.detection_indices:
            detection_to_group[det_idx] = g_idx

    fig, ax = plt.subplots(figsize=(9, 9 * height / width))
    show_image_bgr(ax, sample_image.image_array, title="Overlap groups (each color = one connected group)")

    for i, det in enumerate(trace.detection_result.detections):
        b = det.bbox
        if i in detection_to_group:
            color = group_colors[detection_to_group[i]]
        else:
            color = (0.6, 0.6, 0.6, 1.0)
        rect = mpatches.Rectangle((b.x1, b.y1), b.width, b.height, fill=False, edgecolor=color, linewidth=2)
        ax.add_patch(rect)

    plt.tight_layout()
    plt.show()

    if overlap_result.pairs:
        pairs_df = pd.DataFrame([
            {"detection_a": p.detection_index_a, "detection_b": p.detection_index_b,
             "iou": round(p.iou, 3), "intersection_ratio_a": round(p.intersection_ratio_a, 3),
             "intersection_ratio_b": round(p.intersection_ratio_b, 3), "is_containment": p.is_containment}
            for p in overlap_result.pairs
        ])
        display(pairs_df)
else:
    print("No trace available — skip.")

## 7. Refinement stage — original vs. refined boxes

Gray dashed = original detector bbox, green solid = accepted refinement, orange solid = refinement attempted but rejected by `Refiner._apply_output_policy` (`used_fallback=True` — falls back to the original box; see `src/segmentation/refiner.py`).

In [ ]:
if trace is not None:
    refinement_result = trace.refinement_result
    print(f"triggered={refinement_result.triggered}  backend={refinement_result.backend}  "
          f"refined_boxes={len(refinement_result.refined_boxes)}  "
          f"({refinement_result.processing_time_ms:.1f} ms)")

    if refinement_result.refined_boxes:
        fig, ax = plt.subplots(figsize=(9, 9 * height / width))
        show_image_bgr(ax, sample_image.image_array, title="Refinement: original (gray dashed) vs refined (solid)")

        for rb in refinement_result.refined_boxes:
            original_bbox = trace.detection_result.detections[rb.detection_index].bbox
            orig_rect = mpatches.Rectangle(
                (original_bbox.x1, original_bbox.y1), original_bbox.width, original_bbox.height,
                fill=False, edgecolor="#888888", linewidth=1.2, linestyle="--")
            ax.add_patch(orig_rect)

            color = "#DD8452" if rb.used_fallback else "#55A868"
            b = rb.refined_bbox
            rect = mpatches.Rectangle((b.x1, b.y1), b.width, b.height, fill=False, edgecolor=color, linewidth=2)
            ax.add_patch(rect)
            ax.text(b.x1, b.y1 - 3, f"#{rb.detection_index} cov={rb.mask_area_ratio:.2f}"
                    + (" (fallback)" if rb.used_fallback else ""), color=color, fontsize=7)

        plt.tight_layout()
        plt.show()

        refined_df = pd.DataFrame([
            {"detection_index": rb.detection_index, "mask_area_ratio": round(rb.mask_area_ratio, 3),
             "refinement_confidence": round(rb.refinement_confidence, 3), "backend": rb.backend,
             "used_fallback": rb.used_fallback}
            for rb in refinement_result.refined_boxes
        ])
        display(refined_df)
    else:
        print("No refinement triggered for this image (no suspicious overlap groups found).")
else:
    print("No trace available — skip.")

## 8. Per-crop trace table — Retrieval -> Decision -> Plugins -> Final

Reads `CropTrace` entries directly (`crop`, `retrieval_result`, `decision_result`, `plugin_result`, `final_decision`) — the exact contract `InventoryPipeline._run_stages` builds when `collect_trace=True`.

In [ ]:
if trace is not None and trace.crops:
    crop_rows = []
    for ct in trace.crops:
        crop_rows.append({
            "crop_id": ct.crop.crop_id,
            "detection_index": ct.crop.detection_index,
            "used_refined_bbox": ct.crop.used_refined_bbox,
            "top1_similarity": round(ct.retrieval_result.top_candidate.similarity_score, 4)
                if ct.retrieval_result.top_candidate else None,
            "preliminary_status": ct.decision_result.status,
            "trigger_reasons": sorted(ct.decision_result.trigger_reasons),
            "plugins_ran": ct.plugin_result.executed_plugins if ct.plugin_result else [],
            "final_product_id": ct.final_decision.product_id,
            "final_status": ct.final_decision.status,
            "final_confidence": round(ct.final_decision.final_confidence, 4),
        })
    display(pd.DataFrame(crop_rows))
else:
    print("No crops in trace — skip.")

In [ ]:
if trace is not None and trace.crops:
    reranked_crops = [ct for ct in trace.crops if ct.plugin_result is not None]
    if reranked_crops:
        n = min(len(reranked_crops), 3)
        fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4))
        if n == 1:
            axes = [axes]
        for ax, ct in zip(axes, reranked_crops[:n]):
            title = (f"{ct.decision_result.status} -> {ct.final_decision.status}\n"
                     f"plugins={ct.plugin_result.executed_plugins}")
            show_image_bgr(ax, ct.crop.raw_image_array, title=title)
        plt.tight_layout()
        plt.show()
    else:
        print("No crops required plugin evidence in this trace.")
else:
    print("No trace available — skip.")

## 9. Final annotated result, JSON, and CSV via the real `StorageManager`

Uses `StorageManager.save_all()` directly — the same call `InferenceRunner.run_single` makes — rather than a notebook-local matplotlib rendering, so the annotated image, JSON, and CSV shown here are byte-identical to what production writes to `data/outputs/`.

In [ ]:
if storage is not None and result is not None:
    saved_paths = storage.save_all(sample_image.image_array, result)
    print("Saved artifacts:")
    for name, path in saved_paths.items():
        print(f"  {name}: {path}")

    if "annotated_image" in saved_paths:
        annotated = cv2.imread(saved_paths["annotated_image"])
        fig, ax = plt.subplots(figsize=(9, 9 * height / width))
        show_image_bgr(ax, annotated, title=f"StorageManager.save_annotated_image() output "
                                              f"({result.total_items} item(s))")
        plt.tight_layout()
        plt.show()
else:
    print("Storage or result not available — skip.")

In [ ]:
if storage is not None and "json" in saved_paths:
    with open(saved_paths["json"], "r", encoding="utf-8") as f:
        result_json = json.load(f)
    total_items_val = result_json["total_items"]
    processing_time_val = result_json["processing_time_ms"]
    print(f"result.json top-level keys: {list(result_json.keys())}")
    print(f"total_items: {total_items_val}  processing_time_ms: {processing_time_val:.2f}")

if storage is not None and "csv" in saved_paths:
    csv_df = pd.read_csv(saved_paths["csv"])
    display(csv_df)

## 10. Batch run + per-stage latency breakdown

Runs `run_with_trace()` over a small batch to see how detection/overlap/refinement/retrieval time scales — using each stage's own `processing_time_ms` field rather than wall-clock timing the notebook cell itself.

In [ ]:
if pipeline is not None:
    BATCH_SIZE = min(6, len(available_images))
    batch_paths = random.sample(available_images, k=BATCH_SIZE)

    latency_rows = []
    for path in batch_paths:
        img_array = load_image_bgr(path)
        h, w = img_array.shape[:2]
        img_data = ImageData(image_id=generate_id(prefix="img_"), source_path=str(path),
                              image_array=img_array, width=w, height=h)
        batch_result, batch_trace = pipeline.run_with_trace(img_data)

        latency_rows.append({
            "image": path.name,
            "detections": len(batch_trace.detection_result.detections),
            "overlap_groups": batch_trace.overlap_result.group_count,
            "refinement_triggered": batch_trace.refinement_result.triggered,
            "crops": len(batch_trace.crops),
            "items_accepted": batch_result.total_items,
            "detection_ms": round(batch_trace.detection_result.processing_time_ms, 2),
            "overlap_ms": round(batch_trace.overlap_result.processing_time_ms, 2),
            "refinement_ms": round(batch_trace.refinement_result.processing_time_ms, 2),
            "total_ms": round(batch_result.processing_time_ms, 2),
        })

    latency_df = pd.DataFrame(latency_rows)
    display(latency_df)
else:
    latency_df = pd.DataFrame()
    print("Pipeline not available — skip.")

In [ ]:
if not latency_df.empty:
    stage_cols = ["detection_ms", "overlap_ms", "refinement_ms"]
    fig, ax = plt.subplots(figsize=(9, 4))
    bottom = np.zeros(len(latency_df))
    for col, color in zip(stage_cols, ["#4C72B0", "#DD8452", "#55A868"]):
        ax.bar(latency_df["image"], latency_df[col], bottom=bottom, label=col, color=color)
        bottom += latency_df[col].values
    ax.set_ylabel("Latency (ms)")
    ax.set_title("Per-stage latency breakdown (Detection / Overlap / Refinement)")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    plt.tight_layout()
    plt.show()
    print("Note: total_ms also includes per-crop Retrieval/Decision/Plugin/Rerank time, "
          "not broken out above since it is summed per-crop rather than per-image in the DTOs.")

## 11. Summary

In [ ]:
summary = {
    "Sample image (\u00a74)": sample_path.name if "sample_path" in dir() else "n/a",
    "Detections (\u00a75)": len(trace.detection_result.detections) if trace is not None else "n/a",
    "Overlap groups (\u00a76)": trace.overlap_result.group_count if trace is not None else "n/a",
    "Refinement triggered (\u00a77)": trace.refinement_result.triggered if trace is not None else "n/a",
    "Crops / final accepted items (\u00a78)": f"{len(trace.crops)} / {result.total_items}"
        if trace is not None else "n/a",
    "Batch images processed (\u00a710)": len(latency_df) if not latency_df.empty else 0,
}
pd.DataFrame([summary]).T.rename(columns={0: "Value"})

---
**Quick observations (fill in after running):**

- In §5, are there detector boxes close to `detection.confidence_threshold`? Those are the ones most likely to flip between runs or configs.
- In §6, do the flagged overlap groups actually look like genuinely overlapping/nested products, or does the geometric heuristic (`iou_threshold`/`overlap_ratio_threshold`) sometimes flag adjacent-but-separate items?
- In §7, how often is `used_fallback=True`? A high fallback rate suggests `refinement.output.*` bounds may be too strict for this product line's packaging shapes.
- In §9, does the exported `result.csv` match what you'd expect from manually counting items in the annotated image?
- In §10, which stage dominates latency — is it Detection (RF-DETR) or does Refinement (SAM2) dominate whenever it's triggered?

**Next notebook:** `08_validation_benchmark.ipynb` — running `run.py --mode validate` over the full COCO benchmark and reproducing the README's 9-stage metrics table.